# Network Intrusion Detection System using Transformer - CSE-CIC-IDS2018 Dataset

This notebook applies the same transformer-based multi-class classification methodology
used in `NIDS_Transformer_Multi_Class.ipynb` (KDD Cup 99) to the **CSE-CIC-IDS2018** dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras import layers, Model

%matplotlib inline

## 1. Load CSE-CIC-IDS2018 Dataset

The CSE-CIC-IDS2018 dataset consists of multiple CSV files, one per day of captured traffic.
Update the `data_dir` path below to point to the folder containing the CSV files.

The dataset can be downloaded from:
- [CIC IDS 2018 on AWS](https://www.unb.ca/cic/datasets/ids-2018.html)
- [Kaggle CSE-CIC-IDS2018](https://www.kaggle.com/datasets/solarmainframe/ids-intrusion-csv)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Update this path to point to the folder containing CSE-CIC-IDS2018 CSV files
data_dir = '/content/drive/MyDrive/CSE-CIC-IDS2018/'

In [ ]:
csv_files = glob.glob(os.path.join(data_dir, '*.csv'))
print(f'Found {len(csv_files)} CSV files:')
for f in sorted(csv_files):
    print(f'  {os.path.basename(f)}')

df_list = []
for f in sorted(csv_files):
    temp_df = pd.read_csv(f, low_memory=False)
    print(f'Loaded {os.path.basename(f)}: {temp_df.shape}')
    df_list.append(temp_df)

data = pd.concat(df_list, ignore_index=True)
print(f'\nCombined dataset shape: {data.shape}')

## 2. Data Cleaning

In [ ]:
# Strip whitespace from column names
data.columns = data.columns.str.strip()
print('Columns:', data.columns.tolist())

In [ ]:
# The label column in CSE-CIC-IDS2018 is typically named 'Label'
label_col = 'Label'
print(f'Unique labels ({data[label_col].nunique()}):')
print(data[label_col].value_counts())

In [ ]:
data.drop_duplicates(inplace=True)
print(f'Shape after dropping duplicates: {data.shape}')

In [ ]:
# Replace infinite values with NaN, then drop rows with NaN
data.replace([np.inf, -np.inf], np.nan, inplace=True)
print(f'Missing values per column (top 10):')
print(data.isnull().sum().sort_values(ascending=False).head(10))

data.dropna(inplace=True)
print(f'\nShape after dropping NaN rows: {data.shape}')

## 3. Exploratory Data Analysis

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
label_counts = data[label_col].value_counts()
plt.figure(figsize=(12, 6))
label_counts.plot(kind='bar')
plt.title('Class Distribution - CSE-CIC-IDS2018')
plt.xlabel('Attack Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Feature and Label Preparation

In [ ]:
X = data.drop(label_col, axis=1)

# Drop any remaining non-numeric columns (e.g., Timestamp)
non_numeric_cols = X.select_dtypes(include=['object', 'datetime']).columns.tolist()
if non_numeric_cols:
    print(f'Dropping non-numeric columns: {non_numeric_cols}')
    X = X.drop(columns=non_numeric_cols)

print(f'Feature matrix shape: {X.shape}')

In [ ]:
y_multi = pd.get_dummies(data[label_col])
print(f'Label matrix shape: {y_multi.shape}')
print(f'Classes: {y_multi.columns.tolist()}')

## 5. Train / Validation / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_multi, test_size=0.2, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

In [ ]:
# All features in CSE-CIC-IDS2018 are numeric, so we use MinMaxScaler on all columns
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
print("X_train's shape: ", X_train.shape)
print("y_train's shape: ", y_train.shape)
print("X_val's shape: ", X_val.shape)
print("y_val's shape: ", y_val.shape)
print("X_test's shape: ", X_test.shape)
print("y_test's shape: ", y_test.shape)

In [ ]:
X_train = np.asarray(X_train).astype('float32')
y_train = np.asarray(y_train).astype('float32')
X_val = np.asarray(X_val).astype('float32')
y_val = np.asarray(y_val).astype('float32')

## 6. Transformer Model Architecture

Using the same MultiHeadSelfAttention and TransformerModel as in `NIDS_Transformer_Multi_Class.ipynb`.

In [ ]:
class MultiHeadSelfAttention(layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(MultiHeadSelfAttention, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        assert embed_dim % num_heads == 0
        self.projection_dim = embed_dim // num_heads
        self.query_dense = layers.Dense(embed_dim)
        self.key_dense = layers.Dense(embed_dim)
        self.value_dense = layers.Dense(embed_dim)
        self.combine_heads = layers.Dense(embed_dim)

    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output, weights

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)
        query = self.separate_heads(query, batch_size)
        key = self.separate_heads(key, batch_size)
        value = self.separate_heads(value, batch_size)
        attention, weights = self.attention(query, key, value)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim))
        output = self.combine_heads(concat_attention)
        return output, weights


class TransformerModel(Model):
    def __init__(self, num_heads, ff_dim, num_layers, input_dim, output_dim, dropout_rate=0.1):
        super(TransformerModel, self).__init__()
        self.embedding_layer = layers.Dense(input_dim, activation='relu')
        self.attention = MultiHeadSelfAttention(input_dim, num_heads)
        self.ffn = tf.keras.Sequential(
            [layers.Dense(ff_dim, activation="relu"), layers.Dense(input_dim),]
        )
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)
        self.layer_normal1 = layers.LayerNormalization(epsilon=1e-6)
        self.layer_normal2 = layers.LayerNormalization(epsilon=1e-6)
        self.global_average_pooling = layers.GlobalAveragePooling1D()
        self.fc = layers.Dense(output_dim, activation='softmax')

    def call(self, inputs):
        x = self.embedding_layer(inputs)
        x = self.layer_normal1(x + self.dropout1(self.attention(x)[0]))
        x = self.layer_normal2(x + self.dropout2(self.ffn(x)))
        x = self.global_average_pooling(x)
        return self.fc(x)

## 7. Model Training

In [ ]:
num_heads = 2
ff_dim = 128
num_layers = 2
input_dim = X_train.shape[1]
output_dim = y_multi.shape[1]
dropout_rate = 0.2

print(f'Input dimension: {input_dim}')
print(f'Output dimension (number of classes): {output_dim}')

In [ ]:
model_trans_multi = TransformerModel(num_heads, ff_dim, num_layers, input_dim, output_dim)

loss_fn = tf.keras.losses.CategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model_trans_multi.compile(optimizer=optimizer, loss=loss_fn)

In [ ]:
model_trans_multi.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_val, y_val))

In [ ]:
loss = pd.DataFrame(model_trans_multi.history.history)
loss.plot()
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

## 8. Evaluation

In [ ]:
y_pred_prob = model_trans_multi.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=-1)

print("y_pred_prob.shape ", y_pred_prob.shape)
print("y_pred.shape ", y_pred.shape)
print("y_test.shape ", y_test.shape)

In [ ]:
y_test_array = np.asarray(y_test).astype('float32')
y_test_classes = np.argmax(y_test_array, axis=-1)

In [ ]:
print(classification_report(y_test_classes, y_pred, digits=4, target_names=y_multi.columns.tolist()))

In [ ]:
cm = confusion_matrix(y_test_classes, y_pred)
plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=y_multi.columns.tolist(),
            yticklabels=y_multi.columns.tolist())
plt.title('Confusion Matrix - CSE-CIC-IDS2018 Multi-Class Classification')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()